# The Sigmoid Curve, Binary Cross-Entropy & Gradient Descent Fitting Lab

Fitting a logistic regression model requires optimizing non-linear probabilities via Maximum Likelihood Estimation. This lab explores the manual calculation of Log Loss (Binary Cross-Entropy), contrasts its steep penalty with Mean Squared Error, trains logistic models on synthetic student data, and tracks loss reduction across gradient descent epochs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expit  # Sigmoid function
from sklearn.linear_model import LogisticRegression, SGDClassifier

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Hand-Worked Grid Search over Parameters m and b

Evaluate candidate parameter pairs on a 5-student toy dataset to observe how total log loss changes as the sigmoid curve shifts.

In [ ]:
hours = np.array([2, 4, 6, 8, 10])
passed = np.array([0, 0, 1, 1, 1])

def compute_log_loss(y_true, y_pred):
    eps = 1e-15
    y_p = np.clip(y_pred, eps, 1.0 - eps)
    return -np.mean(y_true * np.log(y_p) + (1 - y_true) * np.log(1 - y_p))

candidates = [
    (0.30, -1.20, "m=0.30, b=-1.20"),
    (0.35, -1.40, "m=0.35, b=-1.40"),
    (0.40, -1.60, "m=0.40, b=-1.60"),
    (0.50, -2.00, "m=0.50, b=-2.00")
]

print(f"{'Candidate':<20} {'Mean Log Loss':<15}")
print("-" * 35)
for m, b, label in candidates:
    p_preds = expit(m * hours + b)
    loss = compute_log_loss(passed, p_preds)
    print(f"{label:<20} {loss:<15.4f}")

## 2. Comparing Loss Penalties: Log Loss vs. Squared Error

Contrast how Binary Cross-Entropy and Squared Error penalize confident mistakes (e.g. true label $y=1$ but predicted $\hat{y}=0.01$).

In [ ]:
y_true = 1
preds = np.array([0.99, 0.90, 0.50, 0.10, 0.01, 0.001])

print(f"{'Prediction (ŷ)':<18} {'Squared Error':<18} {'Log Loss':<18}")
print("-" * 54)
for p in preds:
    se = (y_true - p)**2
    ll = -np.log(max(p, 1e-15))
    print(f"{p:<18.3f} {se:<18.4f} {ll:<18.4f}")

## 3. Stochastic Gradient Descent Convergence Tracking

Train a classifier over 50 epochs using `SGDClassifier(loss='log_loss')` and track the descent of cross-entropy.

In [ ]:
# Generate synthetic dataset with 50 students
X = np.random.uniform(2, 12, 50).reshape(-1, 1)
prob_true = expit((X - 6) / 1.5)
y = (np.random.rand(50, 1) < prob_true).astype(int).ravel()

sgd = SGDClassifier(loss='log_loss', max_iter=1, warm_start=True, random_state=42, learning_rate='constant', eta0=0.05)
losses = []

for epoch in range(50):
    sgd.fit(X, y)
    probs = sgd.predict_proba(X)[:, 1]
    losses.append(compute_log_loss(y, probs))

print(f"Epoch 1  Loss: {losses[0]:.4f}")
print(f"Epoch 10 Loss: {losses[9]:.4f}")
print(f"Epoch 25 Loss: {losses[24]:.4f}")
print(f"Epoch 50 Loss: {losses[-1]:.4f} (Converged)")